#agent structure

1. **Database Access**
   1.1. The agent connects to an Oracle database using **SQLAlchemy**.
   1.2. Only **SELECT queries** are executed to ensure safety and prevent accidental modifications.

2. **Table Selection**
   2.1. An **orchestrator function** decides which table the agent should use for a user’s question.
   2.2. The function sends the question to the **LLM**, which selects the most appropriate table based on context.

3. **Table Context with Pydantic**
   3.1. Tables are represented as attributes in a **Pydantic class**, each corresponding to a database table.
   3.2. The `Field(description=...)` contains information about the table: its purpose, contained data, and what can be returned.
   3.3. These descriptions provide **context for the LLM**, helping it choose the correct table and reduce errors.

4. **Exploring the Selected Table**
   4.1. Once a table is chosen, the agent can perform a **table description (`get_desc`)** to inspect columns, data types, and other metadata.
   4.2. This helps the agent construct queries correctly and understand the table structure.

5. **Data Retrieval (Query Execution)**
   5.1. The agent builds **filtered and limited queries**, always prioritizing:
       - Relevant filters extracted from the user’s question to reduce returned data.
       - Row limits (`LIMIT` or Oracle equivalent) to avoid overloading the LLM’s context.
   5.2. This prevents excessive data consumption and keeps query results manageable.

6. **Execution Pipeline**
   6.1. User question → Table orchestrator chooses table → `get_desc` if necessary → `get_select` with filters and limits → LLM generates final response.
   6.2. The design is modular, safe, and scalable, making it easy to maintain and add new tables.

7. **Best Practices**
   7.1. Never return all rows from large tables.
   7.2. Keep table descriptions up-to-date for accurate LLM context.
   7.3. Validate filters before executing queries.
   7.4. Limit queries to **SELECT only** for security.

In [ ]:
#imports
from pydantic import BaseModel,Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict
from dotenv import load_dotenv
from typing import Annotated, TypedDict, List
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from langgraph.graph import add_messages, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from sqlalchemy import create_engine, String, Integer, Column
from sqlalchemy.orm import declarative_base,sessionmaker
import json
import oracledb

In [ ]:
#agent state

class State(TypedDict):
    messages: Annotated[list,add_messages]
    

In [ ]:
#env config / variables
class Environment(BaseSettings):
    table_1: SecretStr = Field(alias="TABLE1")
    table_2: SecretStr = Field(alias="TABLE2")
    database_user: SecretStr = Field(alias="DATABASE_USER")
    database_password: SecretStr = Field(alias="DATABASE_PASSWORD")
    database_hostname: SecretStr = Field(alias="DATABASE_HOSTNAME")
    database_port: SecretStr = Field(alias="DATABASE_PORT")
    database_service: SecretStr = Field(alias="DATABASE_SERVICE")
    table1_name: SecretStr = Field(alias="DATABASE_TABLE1_NAME")
    table1_pk: SecretStr = Field(alias="DATABASE_TABLE1_PK")
    api_deepseek: SecretStr = Field(description="DeepSeek Api Key")
    model_config = SettingsConfigDict(env_file=".env_database_agent", extra="ignore")

env = Environment()
    

In [ ]:
#class model Tables
class Tables(BaseSettings):
    
    table1: str = Field(
        default=env.table_1.get_secret_value(),
        description="Table that stores alert records across multiple domains, including infrastructure, code, and application errors."
    )
    
    table2: str = Field(
        default=env.table_2.get_secret_value(),
        description="Table containing customer support tickets and service interactions, including requests, issues reported by clients, and records of assistance provided."
    )

In [ ]:
#model choice table
model_llm = ChatDeepSeek(model="deepseek-chat", temperature=0.7, api_key=env.api_deepseek.get_secret_value())
def init_model_llm(input, tables_list):
    prompt = f"""You are an assistant that helps decide **which database table to use** to answer a user’s question.

    You are given:
    1. The user’s question: {input}
    2. A list of available tables, each with:
    - The table name
    - A description of what the table contains
    
    --List of available tables {tables_list}

    Available tables:
    - "Alerts Table": Stores error alerts, logs, and technical failures from infrastructure or applications.
    - "Support Tickets Table": Stores customer support tickets, service requests, and records of assistance provided.

    **Task:** Choose **only one table** that best matches the user’s question.  
    **Expected output:** JSON in the format:
    {{
    "table_name": "<name of the chosen table>"
    }}

    **Examples:**  
    Question: "What application errors occurred yesterday?"  
    Output: {{"table_name": "Alerts Table"}}

    Question: "Which customer support tickets were resolved today?"  
    Output: {{"table_name": "Support Tickets Table"}}

    **Important:**  
    - Return **only the JSON**, nothing else.  
    - Choose only one table."""
    
    response = model_llm.invoke(input=prompt).content

    return response


In [ ]:
@tool
def oracle_conn(query: str):
    """Executes a read-only SQL query against the Oracle database and returns the results.
    Use this tool to retrieve data from a specific Oracle table using a SELECT statement.
    Always apply WHERE filters based on the user's question and include a ROWNUM limit
    (e.g., WHERE ROWNUM <= 50) to avoid returning excessive rows to the LLM context.

    Do NOT use DESC or DESCRIBE commands — use get_table_description tool instead.

    Args:
        query (str): A valid Oracle SQL SELECT statement to be executed.

    Returns:
        list: A list of tuples containing the rows returned by the query.

    Example:
        oracle_conn("SELECT * FROM alerts WHERE ROWNUM <= 10")
    """
    # Intercepta DESC/DESCRIBE e converte para SQL válido
    stripped = query.strip()
    if stripped.upper().startswith("DESC") or stripped.upper().startswith("DESCRIBE"):
        table_name = stripped.split()[-1].strip(";")
        query = f"""
            SELECT COLUMN_NAME, DATA_TYPE, DATA_LENGTH, NULLABLE
            FROM ALL_TAB_COLUMNS
            WHERE TABLE_NAME = UPPER('{table_name}')
            ORDER BY COLUMN_ID
        """

    conn = oracledb.connect(
        user=env.database_user.get_secret_value(),
        password=env.database_password.get_secret_value(),
        dsn=env.database_hostname.get_secret_value() + "/" + env.database_service.get_secret_value()
    )
    with conn.cursor() as session:
        session.execute(query)
        result = session.fetchall()

    return result

In [ ]:
#decide table - tool
@tool
def decide_table(user_question:str):
    """A tool responsible for selecting the most appropriate table to construct SQL queries 
    based on the user’s input question.

    This tool uses table descriptions (from a Pydantic model) as context, 
    allowing the LLM to determine which table contains the relevant data. 
    It outputs only the table name, which will then be used for further actions 
    such as describing the table columns or executing filtered SELECT queries.
"""
    tables = Tables()
    tables_return = []
    
    for table_name in tables:
        tables_return.append({"table_name":table_name[1], "table_description":Tables.model_fields[f"{table_name[0]}"].description})
        
    response = init_model_llm(input=user_question, tables_list=tables_return)
    
    return json.loads(response).get("table_name")

In [ ]:
@tool
def get_table_description(table_name: str):
    """
    Retrieves the schema description of a given Oracle table.

    Queries ALL_TAB_COLUMNS to return column names, data types,
    length, and nullability for the specified table.

    Args:
        table_name (str): The name of the table to be described.

    Returns:
        list: A list of tuples with (COLUMN_NAME, DATA_TYPE, DATA_LENGTH, NULLABLE).
    """
    conn = oracledb.connect(
        user=env.database_user.get_secret_value(),
        password=env.database_password.get_secret_value(),
        dsn=env.database_hostname.get_secret_value() + "/" + env.database_service.get_secret_value()
    )
    with conn.cursor() as session:
        session.execute(f"""
            SELECT COLUMN_NAME, DATA_TYPE, DATA_LENGTH, NULLABLE
            FROM ALL_TAB_COLUMNS
            WHERE TABLE_NAME = UPPER('{table_name}')
            ORDER BY COLUMN_ID
        """)
        result = session.fetchall()
    return result

In [ ]:
#model llm with tools
model_llm_tools = ChatDeepSeek(model="deepseek-chat", temperature=0.7, api_key=env.api_deepseek.get_secret_value()).bind_tools([decide_table, oracle_conn,get_table_description])

In [ ]:
tools = [decide_table, oracle_conn, get_table_description]
tool_node = ToolNode(tools)

In [ ]:
def chatbot(state: State):
    response = model_llm_tools.invoke(state['messages'])
    return {"messages":response}

In [ ]:
#function to decide tool execution
def should_tool_call(state: State):
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "tools"
    return END

In [ ]:
#graph pipeline
builder = StateGraph(State)

#nodes
builder.add_node("chatbot",chatbot)
builder.add_node("tools", tool_node)

#entry point
builder.set_entry_point("chatbot")

#conditional edge
builder.add_conditional_edges("chatbot", should_tool_call)
builder.add_edge("tools","chatbot")

#memory - checkpointer
memory = InMemorySaver()

#compile pipeline
graph = builder.compile(checkpointer=memory)

#session id manual
settings = {"configurable":{"thread_id": "1"}}

In [ ]:
if __name__ == "__main__":
    user_input = input("type agent :")
    result = []
    while user_input.lower().strip() not in ['exit','sair']:
        result.append(graph.invoke({"messages":[{"role":"system","content":"You are a helpful assistant with access to an Oracle database. Always use the available tools to answer questions. Keep your responses simple and fast.","role":"user","content":user_input}]}, config=settings))
        print(result[-1]['messages'][-1].content)
        user_input = input("type agent: ")